# MPC Negative Observations API

#### This tutorial demonstrates how to report a *negative observation*, i.e. "We imaged this field and did **not** detect the object", to the Minor Planet Center.

Negative observations are valuable science: knowing that a known or suspected object was *not* seen down to a stated limiting magnitude helps constrain orbits (especially for NEOCP candidates), rule out identifications, and calibrate survey performance. See the [negative observations documentation](https://docs.minorplanetcenter.net/mpc-ops-docs/observations/negative-observations/) for the scientific background.

A negative observation is submitted as a **pointing record with extra fields**: the payload describes the exposure exactly as in the [Pointings API tutorial](mpc_tutorial_api_pointings.ipynb), and adds fields stating which object was looked for, that it was not found, and (optionally) details of how the limiting magnitude was estimated.

Further information and documentation can be found at:
 - [Pointing submissions documentation](https://docs.minorplanetcenter.net/services/negative_observations/) (field definitions for pointings and negative observations)
 - [Negative observations documentation](https://docs.minorplanetcenter.net/mpc-ops-docs/observations/negative-observations/)


# Import Packages
Here we import the standard python packages we use in this tutorial

In [2]:
import requests
import json

# API Parameters

A negative observation is submitted as a JSON payload via a POST request using `requests.post(url, json=payload)`.

## Pointing fields

The payload contains all of the usual pointing fields — see the [Pointings API tutorial](mpc_tutorial_api_pointings.ipynb) for the full tables:

- **Required:** `action` (`"exposed"`), `surveyExpName`, `mode`, `mpcCode`, `time`, `duration`, `center`, `limit`
- **Field geometry (exactly one):** `width`, `widths`, `fieldDiam`, or `offsets`
- **Optional:** `filter`, `nonsidereal`

## Additional required fields for negative observations

| Field | Type | Description |
|-------|------|-------------|
| `found` | Boolean | `false` for a negative observation |
| `desig` | String | Designation of the object searched for, in [packed format](https://docs.minorplanetcenter.net/mpc-ops-docs/designations/packed-designations/), or a `trksub` for an NEOCP candidate |
| `submitter` | String | Name of the submitter |

## Additional optional fields

| Field | Description |
|-------|-------------|
| `limiting_mag_method` | Integer in {1, 2, 3, 4} identifying how the limiting magnitude was estimated (see below) |
| `notes` | Free-text notes |
| `number_of_stars_fov` | Number of stars in the field of view |
| `pixel_scale` | Pixel scale of the detector |
| `seeing` | Seeing estimate |
| `software` | Software used |
| `stacked` | Whether frames were stacked |
| `fill_factor` | Fraction of the field of view covered by active detector area |

## Limiting magnitude methods

1. On a star stack, measure faint objects at the limit and scale to 5-sigma.
2. On a stack, measure the sky noise, scale for a point-source area, and then to 5-sigma.
3. Insert artificial objects into the frames and measure the 50 percent detection efficiency.
4. Manual/custom method.


# API Endpoint

<span style="color: red;">**Warning:** The Negative Observations API has no test endpoint. Every POST request inserts a real record into the MPC Pointings Database. Do not submit example data.</span>

In [3]:
# URL of the Negative Observations API endpoint
url = "https://www.minorplanetcenter.net/cgi-bin/pointings/submit_negativeObs"

# Example 1: Negative Observation of an NEOCP Candidate

This example reports a targeted follow-up exposure in which an NEOCP candidate (identified by its `trksub`, here `"P11Uypl"`) was searched for but **not found** down to a limiting magnitude of 22.7.

<span style="color: red;">To prevent accidental submission of example data into the MPC's production database, the submission code a couple of cells below is displayed as markdown rather than an executable code cell.</span>

In [4]:
# Build the JSON payload for a negative observation of an NEOCP candidate
payload_negative = {
    "action": "exposed",
    "surveyExpName": "20180101-EX0132",
    "mode": "target",
    "mpcCode": "I52",
    "time": "2024-03-03T12:20:33.46",
    "duration": 45,
    "center": [255.167, -29.008],
    "widths": [0.82, 0.64],
    "desig": "P11Uypl",
    "limit": 22.7,
    "nonsidereal": True,
    "filter": "V",
    "found": False,
    "submitter": "A. Tomatic",
    "fill_factor": 0.98
}

# Display the JSON payload
print(json.dumps(payload_negative, indent=2))

{
  "action": "exposed",
  "surveyExpName": "20180101-EX0132",
  "mode": "target",
  "mpcCode": "I52",
  "time": "2024-03-03T12:20:33.46",
  "duration": 45,
  "center": [
    255.167,
    -29.008
  ],
  "widths": [
    0.82,
    0.64
  ],
  "desig": "P11Uypl",
  "limit": 22.7,
  "nonsidereal": true,
  "filter": "V",
  "found": false,
  "submitter": "A. Tomatic",
  "fill_factor": 0.98
}


Submit this negative observation to the API:

<span style="color: red;">This cell is deliberately set to markdown format (rather than code) to prevent accidental submission of example data into the MPC's production database.</span>

```python
"""
This cell deliberately set to `markdown` format (rather than `code`) to reduce the risk of
accidental submission of example data into the MPC's production system.
"""

# Submit the negative observation
response = requests.post(url, json=payload_negative)

# Print the response
print(response.status_code)
print(response.reason)
print(response.text)
```

# Example 2: Negative Observation with Detection-Efficiency Metadata

This example reports a negative observation of a designated object (packed designation `"K18A00A"`) and includes the optional metadata fields describing how the limiting magnitude was estimated — here method 3 (artificial objects inserted into the frames, 50 percent detection efficiency) — along with the observing conditions and software used.

<span style="color: red;">To prevent accidental submission of example data into the MPC's production database, the submission code a couple of cells below is displayed as markdown rather than an executable code cell.</span>

In [4]:
# Build the JSON payload with optional limiting-magnitude metadata
payload_negative_meta = {
    "action": "exposed",
    "surveyExpName": "20240303-EX0201",
    "mode": "target",
    "mpcCode": "I52",
    "time": "2024-03-03T13:05:10.12",
    "duration": 120,
    "center": [141.502, 12.337],
    "fieldDiam": 1.1,
    "desig": "K18A00A",
    "limit": 23.1,
    "filter": "r",
    "found": False,
    "submitter": "A. Tomatic",
    "limiting_mag_method": 3,
    "seeing": 1.4,
    "pixel_scale": 0.47,
    "number_of_stars_fov": 1450,
    "software": "custom-pipeline",
    "stacked": True,
    "fill_factor": 0.95,
    "notes": "Three 40s frames stacked on the predicted motion; no candidate at the ephemeris position."
}

# Display the JSON payload
print(json.dumps(payload_negative_meta, indent=2))

{
  "action": "exposed",
  "surveyExpName": "20240303-EX0201",
  "mode": "target",
  "mpcCode": "I52",
  "time": "2024-03-03T13:05:10.12",
  "duration": 120,
  "center": [
    141.502,
    12.337
  ],
  "fieldDiam": 1.1,
  "desig": "K18A00A",
  "limit": 23.1,
  "filter": "r",
  "found": false,
  "submitter": "A. Tomatic",
  "limiting_mag_method": 3,
  "seeing": 1.4,
  "pixel_scale": 0.47,
  "number_of_stars_fov": 1450,
  "software": "custom-pipeline",
  "stacked": true,
  "fill_factor": 0.95,
  "notes": "Three 40s frames stacked on the predicted motion; no candidate at the ephemeris position."
}


Submit this negative observation to the API:

<span style="color: red;">This cell is deliberately set to markdown format (rather than code) to prevent accidental submission of example data into the MPC's production database.</span>

```python
"""
This cell deliberately set to `markdown` format (rather than `code`) to reduce the risk of
accidental submission of example data into the MPC's production system.
"""

# Submit the negative observation
response = requests.post(url, json=payload_negative_meta)

# Print the response
print(response.status_code)
print(response.reason)
print(response.text)
```

# Notes on Responses and Common Errors

A successful submission returns a JSON response with the inserted record ID, in the same style as the Pointings API.

Common reasons for failed submissions include:

- Missing the negative-observation fields (`found`, `desig`, `submitter`)
- Missing required pointing fields (e.g. `surveyExpName`, `mpcCode`, `time`, `duration`, `center`, `limit`)
- No field geometry specified, or multiple geometry fields provided (must be exactly one of `width`, `widths`, `fieldDiam`, `offsets`)
- Invalid `time` format (must be `YYYY-MM-DDThh:mm:ss.sss`)
- Spaces in `surveyExpName`
- `desig` not in packed format (or not a valid NEOCP `trksub`)

You can also submit negative observations using cURL:

```bash
curl -X POST -H "Content-Type: application/json" \
  -d @json.txt https://www.minorplanetcenter.net/cgi-bin/pointings/submit_negativeObs
```

# Summary

This tutorial demonstrated how to use the MPC Negative Observations API to report that an object was searched for but not detected.

- **API endpoint**: `https://www.minorplanetcenter.net/cgi-bin/pointings/submit_negativeObs`
- **Method**: `requests.post(url, json=payload)`
- **Payload**: a standard pointing record (see the [Pointings API tutorial](mpc_tutorial_api_pointings.ipynb)) plus `found: false`, the `desig` searched for (packed format, or a `trksub` for NEOCP candidates), and the `submitter` name
- **Optional metadata**: `limiting_mag_method` (1-4), `seeing`, `pixel_scale`, `number_of_stars_fov`, `software`, `stacked`, `fill_factor`, `notes`

For more information, see:
 - [Pointing submissions documentation](https://docs.minorplanetcenter.net/services/negative_observations/)
 - [Negative observations documentation](https://docs.minorplanetcenter.net/mpc-ops-docs/observations/negative-observations/)
 - [Pointings API tutorial](mpc_tutorial_api_pointings.ipynb)

For questions or feedback, contact the MPC via the [Jira Helpdesk](https://mpc-service.atlassian.net/servicedesk/customer/portal/13/create/148).